In [16]:
import jax.numpy as jnp
import jax
import jax.random as random
from flax import linen as nn
import tensorflow as tf
import optax
from tqdm import tqdm

In [44]:
key = random.PRNGKey(0) # chiave per random 

f_to_learn = lambda mu, k, l, x: jnp.sin(2*mu*jnp.pi*x) + k + jnp.exp(l*x)
N = 10000

key, subkey = random.split(key) # ogni volta, prima di usare la chiave, la devi dividere
x = random.uniform(subkey, (N,), minval=-10, maxval=10)
key, subkey = random.split(key)
mu = random.uniform(subkey, (N,), minval=-2, maxval=2)
key, subkey = random.split(key)
k = random.uniform(subkey, (N,), minval=-5, maxval=5)
key, subkey = random.split(key)
l = random.uniform(subkey, (N,), minval=-1, maxval=1)

y = f_to_learn(mu, k, l, x) # così generiamo artificialmente un dataset di N punti

In [45]:
X = jnp.stack([mu, k, l, x], axis=1)

In [46]:
X.shape
y.shape

(10000,)

In [ ]:
X = jnp.stack([mu, k, l, x], axis=1)


split_idx = int(N * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

train_dataset = train_dataset.shuffle(buffer_size=split_idx).batch(32)
test_dataset = test_dataset.batch(32)

print(train_dataset)
print(test_dataset)

<_BatchDataset element_spec=(TensorSpec(shape=(None, 4), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))>
<_BatchDataset element_spec=(TensorSpec(shape=(None, 4), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))>


In [21]:
class MLP(nn.Module):
    """
    Author: RICCARDO ROTA, ASSOLUTAMENTE LEONARDO BOCCHIERI NON HA CONTRIBUITO.
    """
    # attributi che vengono inizializzati nell'init di nn.Module, quindi andranno passati in input quando inizializziamo il modello, 
    # tipo model = MLP(output_dim=1, hidden_dim=8, num_hidden_layers=2), oppure semplicemente MLP() perchè ho messo default
    output_dim: int = 1
    hidden_dim: int = 8
    num_hidden_layers: int = 2

    @nn.compact
    def __call__(self, x):
        for _ in range(self.num_hidden_layers):
            x = nn.Dense(self.hidden_dim)(x) # strato dense (fully connected), capisce la dimensione dell'input 
                                             # da solo, gli dobbiamo specificare quella dell'output
            x = nn.relu(x) # vedi di capire da solo cos'è questo
        x = nn.Dense(self.output_dim)(x) 
        return x

In [22]:
targetnetwork = MLP(output_dim=1, hidden_dim=8, num_hidden_layers=1)

In [23]:
x = jnp.ones((1,1)) #Gli input sono SEMPRE (SEMPRE) nel formato (bathc_size, input_dim1, input_dim2, ..., input_dimN)
# In questo caso, batch_size=1, input_dim=1
key = jax.random.PRNGKey(0) # Bisogna sempre passare una key per inizializzare i pesi random
print(targetnetwork.tabulate(key, x)) # Visualizza la struttura del modello, con i pesi inizializzati


                               MLP Summary                               
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ path    ┃ module ┃ inputs       ┃ outputs      ┃ params               ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│         │ MLP    │ float32[1,1] │ float32[1,1] │                      │
├─────────┼────────┼──────────────┼──────────────┼──────────────────────┤
│ Dense_0 │ Dense  │ float32[1,1] │ float32[1,8] │ bias: float32[8]     │
│         │        │              │              │ kernel: float32[1,8] │
│         │        │              │              │                      │
│         │        │              │              │ 16 (64 B)            │
├─────────┼────────┼──────────────┼──────────────┼──────────────────────┤
│ Dense_1 │ Dense  │ float32[1,8] │ float32[1,1] │ bias: float32[1]     │
│         │        │              │              │ kernel: float32[8,1] │
│         │        │              │  

In [24]:
hypernetwork = MLP(output_dim = 25, hidden_dim=8, num_hidden_layers=2) # 25 come i parametri del target network

## First try: only training a single network

In [41]:
batch_size = train_dataset.element_spec[0]

In [42]:
print(batch_size)

TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)


In [36]:
model = MLP(output_dim=1, hidden_dim=8, num_hidden_layers=1)

In [49]:
def mse_loss(preds, targets):
    return jnp.mean((preds - targets) ** 2)

optimizer = optax.adam(learning_rate=1e-3)
params = model.init(jax.random.PRNGKey(0), jnp.zeros((1, 4)))
opt_state = optimizer.init(params)
epochs = 100

In [ ]:
@jax.jit
def train_step(params, opt_state, loss_fn, optimizer, x, y, key):
    losses, grads = jax.value_and_grad(loss_fn)(params, x, y, key)
    loss, (mse_loss, kl_loss) = losses
    
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)

    return params, opt_state, loss, mse_loss, kl_loss

for _ in range(epochs):
    for batch in tqdm(train_dataset):
        params, loss = train_step(params, batch)
        print(f"Loss: {loss:.4f}")

#NON FUNZIONA PERCHE PASSIAMO DATASET DI TENSORFLOW E JAX.JIT NON LI CONOSCE, DOBBIAMO RIFARLI IN NUMPY

  0%|          | 0/250 [00:00<?, ?it/s]


TypeError: Error interpreting argument to <function train_step at 0x7700766e65c0> as an abstract array. The problematic value is of type <class 'tensorflow.python.framework.ops.EagerTensor'> and was passed to the function at path args[1][0].
This typically means that a jit-wrapped function was called with a non-array argument, and this argument was not marked as static using the static_argnums or static_argnames parameters of jax.jit.

In [ ]:
from flax.training import train_state

key, model_key = random.split(key)
params = targetnetwork.init(model_key, jnp.zeros((1, 1)))



https://huggingface.co/blog/afmck/flax-tutorial
https://wandb.ai/jax-series/simple-training-loop/reports/Writing-a-Training-Loop-in-JAX-and-Flax--VmlldzoyMzA4ODEy